> **Deprecated.** Superseded by `analysis/make_fig_slope.py` (Fig. 3).
>
> mu labels were the uncorrected vacuum-fraction estimate from one repetition, and one panel-b histogram was shifted by 2 bins by hand (`np.roll`). The script uses the calibrated mu, no shift, and the common display coordinates of `analysis/display.py`.
>
> Kept unchanged (outputs stripped) as a record of the original 2023 analysis; it is not maintained.

# Fig. 3: slope correction (20 GHz)

Cell `slope_cancellation` makes `../figs/slope_cancellation_{light,dark}.{pdf,svg}` (manuscript Fig. 3) from the preamble data in `../inter/decode_20GHz{22,26,30}.0.json`.

Note: the mu labels are the uncorrected vacuum-fraction estimates (repetition 1 only); the 30 dB histogram in panel b is shifted by 2 bins (`np.roll`), and panel c shows the corrected t_B histogram, as in the published figure.


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from snsphd import viz
import math
import orjson
Colors, pallet = viz.phd_style(jupyterStyle = True, grid = True, axese_width=0.5, text = 6, svg_mode=True)

from enum import Enum
from dataclasses import dataclass, fields, field, _MISSING_TYPE 

from analysis.load_schema import Decode, Result

import matplotlib as mpl
import os

# set the DPI for all figures
mpl.rcParams['figure.dpi'] = 100

from snsphd.layout import bisect

In [ ]:



with open("../inter/decode_20GHz26.0.json", "r") as f:
    data = orjson.loads(f.read())
    decode = Decode(**data)

db_values = np.arange(16,50, 2).tolist()
print(db_values)

# need type hints for intellisense to work
def decode_from_folder(folder_path: str, db_values: list[int]) -> list[Decode]:
    decodes = []
    for db in db_values:
        filename = f"decode_20GHz{db}.0.json"
        filepath = os.path.join(folder_path, filename)
        with open(filepath, "r") as f:
            data = orjson.loads(f.read())
            decode = Decode(**data)
            decodes.append(decode)
    return decodes

decodes = decode_from_folder("../inter", db_values)

In [ ]:
for item in decode.results:
    print(len(item))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots()

# Create a list of colors using the viridis colormap
colors = plt.cm.viridis(np.linspace(0, 1, len(decodes)))

for d in range(len(decodes)):
    ax.plot([decodes[d].gm_data.gm_list[i].log_likelihood for i in range(len(decodes[d].gm_data.gm_list))],
            c=colors[d], label=f"{db_values[d]} dB")

# Add a colorbar to the plot
sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis, norm=plt.Normalize(vmin=0, vmax=len(decodes)-1))
sm.set_array([])
fig.colorbar(sm, ax=ax)

ax.legend()


In [ ]:
db_values

In [ ]:
d=12
plt.plot(decodes[d].correction_data.corrected_bins[:-1], decodes[d].correction_data.corrected_hist1, label=f"{db_values[d]} dB")

d=8
plt.plot(decodes[d].correction_data.corrected_bins[:-1], decodes[d].correction_data.corrected_hist1, label=f"{db_values[d]} dB")

d=4
plt.plot(decodes[d].correction_data.corrected_bins[:-1], decodes[d].correction_data.corrected_hist1, label=f"{db_values[d]} dB")

d=0
plt.plot(decodes[d].correction_data.corrected_bins[:-1], decodes[d].correction_data.corrected_hist1, label=f"{db_values[d]} dB")
plt.xlim(-100, 100)
plt.legend()

In [ ]:
# plt.plot(decode.hist_data.bins[:-1], decode.hist_data.counts)
# from IPython.display import Image
# viz.matplotlib_ipython_png_mode()
import matplotlib
from matplotlib.colors import LogNorm

# matplotlib.use('png')
# _ = ax.hist2d(decode.hist_data.counts[:,0],decode.hist_data.counts[:,1], bins=(decode.hist_data.bins, decode.hist_data.bins), cmap="viridis")

d = decode.hist_data

hist, xedges, yedges = np.histogram2d(
    d.counts[:, 1], d.counts[:, 0], bins=(decode.hist_data.bins, decode.hist_data.bins)
)

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(
    hist,
    extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
    norm=LogNorm(),
    cmap="viridis_r",
    origin="lower", rasterized=True)

ax.set_xlim(-700, -500)
ax.set_ylim(-250, 250)


In [ ]:
# len(decodes[3].results[0])

# decodes[3].results[0][1].result


# @dataclass
# class PoissonData:
#     zeros: int = 0
#     one_plus: int = 0


mfrs = []
for d in range(len(decodes)):

    missing = 0
    found = 0
    for res in decodes[d].results[0]:
        result: Result = res.result
        if result == Result.MISSING:
            missing += 1
        if (result == Result.CORRECT) or (result == Result.INCORRECT):
            found += 1
        
    total = missing + found

    ratio_of_vacuum = missing/total
    mfr = -np.log(ratio_of_vacuum)
    mfrs.append(mfr)


# print("total: ", total, "real_total: ", len(decodes[3].results[0]))
# print("mfr", mfr)
print("mrfs", mfrs)


In [ ]:
import shutil
# the panel labels use LaTeX (\bm) if a LaTeX installation is available, matplotlib mathtext otherwise
USETEX = shutil.which("latex") is not None
plt.rcParams['text.latex.preamble'] = r'\usepackage{bm}'

idx = 5

d = decodes[idx].hist_data
# d = decode.hist_data

left, right = bisect([0,0,1,1], direction='vert', offset=0.5, spacing=0.12)
left_bottom, left_top = bisect(left, direction='horiz', offset=0.5, spacing=0.12)

fig = plt.figure(figsize=(11, 5))


left_top_ax = fig.add_axes(left_top)
left_bottom_ax = fig.add_axes(left_bottom)


right_ax = fig.add_axes(right)

# left_top_ax.plot(d.slices[:-1], d.corr1-np.average(d.counts[:, 0]), color='black')
left_top_ax.plot(d.slices[:-1], d.corr1, color='black')
print(d.bins)
left_top_ax.hist2d(
            d.counts[:, 1] - d.counts[:, 0],
            d.counts[:, 0],
            bins=(d.bins, d.bins),
            norm=matplotlib.colors.LogNorm(), rasterized=True, alpha=0.5)
# left_top_ax.hist2d(
#             d.counts[:, 1] - d.counts[:, 0],
#             d.counts[:, 0]-np.average(d.counts[:, 0]),
#             bins=(d.bins, d.bins),
#             norm=matplotlib.colors.LogNorm(), rasterized=True, alpha=0.5)
left_top_ax.text(0.05, 0.95, f"$\mu$ = {mfrs[idx]:.2f}", transform=left_top_ax.transAxes, ha='left', va='top', color='black', fontsize=15)

print(np.average(d.counts[:, 0]))
left_top_ax.set_xlim(500, 750)
left_top_ax.set_ylim(np.average(d.counts[:, 0])-100, np.average(d.counts[:, 0])+100)

left_top_ax.set_xlabel(r"$\Delta t_{BA}$ (ps)")
left_top_ax.set_ylabel("$t_A$ (ps)", labelpad=0)




colors = plt.cm.viridis(np.linspace(0, 1, 4))

idx = 7
hist, bins = np.histogram(decodes[idx].hist_data.counts[:, 1] - decodes[idx].hist_data.counts[:, 0], bins=d.bins)
left_bottom_ax.plot(bins[:-1], np.roll(hist,2), color=colors[0], label=f"$\mu$ = {mfrs[idx]:.2f}", alpha=0.7)

idx = 5
hist, bins = np.histogram(decodes[idx].hist_data.counts[:, 1] - decodes[idx].hist_data.counts[:, 0], bins=d.bins)
left_bottom_ax.plot(bins[:-1], hist, color=colors[1], label=f"$\mu$ = {mfrs[idx]:.2f}", alpha=0.7)

idx = 3
hist, bins = np.histogram(decodes[idx].hist_data.counts[:, 1] - decodes[idx].hist_data.counts[:, 0], bins=d.bins)
left_bottom_ax.plot(bins[:-1], hist, color=colors[2], label=f"$\mu$ = {mfrs[idx]:.2f}", alpha=0.7)
leg = left_bottom_ax.legend(fancybox=False, framealpha=0.1)
leg.get_frame().set_linewidth(1.5)
# left_bottom_ax.set_yscale("log")
left_bottom_ax.set_xlabel(r"$\Delta t_{BA}$ (ps)")
left_bottom_ax.set_ylabel("counts")


left_bottom_ax.set_xlim(500, 750)
right_ax_start = -78
right_ax_end = 78
right_ax.set_xticks([-75, -50, -25, 0, 25, 50, 75])

right_ax.plot(decodes[7].correction_data.corrected_bins[:-1], decodes[7].correction_data.corrected_hist2, c=colors[0], label=f"$\mu$ = {mfrs[7]:.2f}", alpha=0.7)
right_ax.plot(decodes[5].correction_data.corrected_bins[:-1], decodes[5].correction_data.corrected_hist2, c=colors[1], label=f"$\mu$ = {mfrs[5]:.2f}", alpha=0.7)
right_ax.plot(decodes[3].correction_data.corrected_bins[:-1], decodes[3].correction_data.corrected_hist2, c=colors[2], label=f"$\mu$ = {mfrs[3]:.2f}", alpha=0.7)



leg = right_ax.legend(fancybox=False, framealpha=0.1)
leg.get_frame().set_linewidth(1.5)
right_ax.axvline(-25, color='red', alpha=0.5)
right_ax.axvline(25, color='red', alpha=0.5)
right_ax.axvspan(-25, right_ax_start, color='red', alpha=0.05)
right_ax.axvspan(25, right_ax_end, color='red', alpha=0.05)
right_ax.set_xlim(right_ax_start, right_ax_end)
right_ax.set_ylabel("counts")
right_ax.set_xlabel(r"$\tilde{t_A}$ (ps)")

left_top_ax.text(-.10, 1.08, "a)", transform=left_top_ax.transAxes, fontsize=19, va='top') # require import shutil
# the panel labels use LaTeX (\bm) if a LaTeX installation is available, matplotlib mathtext otherwise
USETEX = shutil.which("latex") is not None
plt.rcParams['text.latex.preamble'] = r'\usepackage{bm}'
left_bottom_ax.text(-.10, 1.08, "b)", transform=left_bottom_ax.transAxes, fontsize=19, va='top')
right_ax.text(-.10, 1.05, "c)", transform=right_ax.transAxes, fontsize=19, va='top')

ket = (lambda n: rf'$\bm{{|{n}\rangle}}$') if USETEX else (lambda n: rf'$\mathbf{{|{n}\rangle}}$')
left_top_ax.text(650, -510, ket('1'), fontsize=16, usetex=USETEX)
left_top_ax.text(580, -560, ket('2'), fontsize=16, usetex=USETEX)
left_top_ax.text(540, -590, ket('3+'), fontsize=16, usetex=USETEX)

viz.save_light_dark_all(fig, name="../figs/slope_cancellation", override_alpha=True)


In [ ]:
1 - (10/400)

In [ ]:
400/(10 + 400)